## Imports

In [ ]:
import tensorflow as tf

import os
import datetime
import time

import pyvista as pv
from matplotlib import pyplot as plt
from IPython import display
import numpy as np

# `pix2pix` implementation

### Load .vti files and convert to tensor

In [ ]:
def resize(input_image, real_image=None, height=256, width=256):
  input_image = tf.expand_dims(input_image, axis=-1)

  input_image = tf.image.resize(input_image, [height, width],
                                method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)

  if real_image != None:
    real_image = tf.expand_dims(real_image, axis=-1)
    real_image = tf.image.resize(real_image, [height, width],
                                method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)

    return input_image, real_image

  return input_image, None

def load(input_file_path, real_file_path=None):
    input_grid = pv.read(input_file_path)
    input_data = input_grid.active_scalars
    input_dims = input_grid.dimensions
    input_nx, input_ny = input_dims[0], input_dims[1]

    input_image = input_data.reshape((input_ny, input_nx))
    input_tensor = tf.convert_to_tensor(input_image, dtype=tf.float32)

    if real_file_path:
      real_grid = pv.read(real_file_path)
      real_data = real_grid.active_scalars
      real_dims = real_grid.dimensions
      real_nx, real_ny = real_dims[0], real_dims[1]

      real_image = real_data.reshape((real_ny, real_nx))
      real_tensor = tf.convert_to_tensor(real_image, dtype=tf.float32)

      return resize(input_tensor, real_tensor, 256, 256)

    return resize(input_tensor, None, 256, 256)

In [ ]:
inp, re = load('../sample/constant_vel/coarse_grid/output_data_0_15.vti', '../sample/constant_vel/refined_grid/output_data_0_15.vti')

plt.figure()
plt.imshow(inp, cmap='grey', vmin=-.5, vmax=.5)
plt.figure()
plt.imshow(re, cmap='grey', vmin=-.5, vmax=.5)

### Split data into training/testing

In [ ]:
# ! bash ../scripts/split_data.sh ../sample/constant_vel/coarse_grid "output_data_0" "input_1"
# ! bash ../scripts/split_data.sh ../sample/constant_vel/refined_grid "output_data_0" "real_1"

# ! bash ../scripts/split_data.sh ../sample/parallel_planes/coarse_grid "output_data_0" "input_2"
# ! bash ../scripts/split_data.sh ../sample/parallel_planes/refined_grid "output_data_0" "real_2"

! bash ../scripts/split_data.sh ../sample/semi_circle/coarse_grid "output_data_0" "input_3"
! bash ../scripts/split_data.sh ../sample/semi_circle/refined_grid "output_data_0" "real_3"

### Building input Pipeline

In [ ]:
def load_from_tensor(input_path_tensor, real_path_tensor):
    return load(input_path_tensor.numpy().decode('utf-8'), real_path_tensor.numpy().decode('utf-8'))

def load_wrapper(input_path, real_path):
    input_tensor, real_tensor = tf.py_function(
        func=load_from_tensor,
        inp=[input_path, real_path],
        Tout=[tf.float32, tf.float32]
    )
    # input_tensor.set_shape([256, 256])  # adjust based on actual shape
    # real_tensor.set_shape([256, 256])
    return input_tensor, real_tensor

def make_dataset(dir_path, batch_size=16, shuffle=True, repeat=False):
    # Get all input and real files sorted to match
    input_files = sorted(tf.io.gfile.glob(os.path.join(dir_path, "input_*.vti")))
    real_files = sorted(tf.io.gfile.glob(os.path.join(dir_path, "real_*.vti")))
    # input_files = sorted([os.path.join(dir_path, f) for f in os.listdir(dir_path) if f.startswith('input_')])
    # real_files = sorted([os.path.join(dir_path, f) for f in os.listdir(dir_path) if f.startswith('real_')])

    dataset = tf.data.Dataset.from_tensor_slices((input_files, real_files))

    dataset = dataset.map(load_wrapper, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        dataset = dataset.shuffle(buffer_size=32)

    if repeat:
        dataset = dataset.repeat()

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

    return dataset


In [ ]:
train_dataset = make_dataset("../train", batch_size=1)
test_dataset = make_dataset("../test", batch_size=1, shuffle=False)

### Build the generator

In [ ]:
OUTPUT_CHANNELS = 1

In [ ]:
def downsample(filters, size, apply_batchnorm=True):
  initializer = tf.random_normal_initializer(0., 0.02)

  result = tf.keras.Sequential()
  result.add(
      tf.keras.layers.Conv2D(filters, size, strides=2, padding='same',
                             kernel_initializer=initializer, use_bias=False))

  if apply_batchnorm:
    result.add(tf.keras.layers.BatchNormalization())

  result.add(tf.keras.layers.LeakyReLU())

  return result

In [ ]:
down_model = downsample(3, 4)
down_result = down_model(tf.expand_dims(inp, 0))
print(down_result.shape)

In [ ]:
def upsample(filters, size, apply_dropout=False):
  initializer = tf.random_normal_initializer(0., 0.02)

  result = tf.keras.Sequential()
  result.add(
    tf.keras.layers.Conv2DTranspose(filters, size, strides=2,
                                    padding='same',
                                    kernel_initializer=initializer,
                                    use_bias=False))

  result.add(tf.keras.layers.BatchNormalization())

  if apply_dropout:
      result.add(tf.keras.layers.Dropout(0.5))

  result.add(tf.keras.layers.ReLU())

  return result

In [ ]:
# def Generator():
#   inputs = tf.keras.layers.Input(shape=[256, 256, 1])

#   down_stack = [
#     downsample(64, 2, apply_batchnorm=False),  # (batch_size, 128, 128, 64)
#     downsample(128, 2),  # (batch_size, 64, 64, 128)
#     downsample(256, 2),  # (batch_size, 32, 32, 256)
#     downsample(512, 2),  # (batch_size, 16, 16, 512)
#     downsample(512, 2),  # (batch_size, 8, 8, 512)
#     downsample(512, 2),  # (batch_size, 4, 4, 512)
#     downsample(512, 2),  # (batch_size, 2, 2, 512)
#     downsample(512, 2),  # (batch_size, 1, 1, 512)
#   ]

#   up_stack = [
#     upsample(512, 2, apply_dropout=True),  # (batch_size, 2, 2, 1024)
#     upsample(512, 2, apply_dropout=True),  # (batch_size, 4, 4, 1024)
#     upsample(512, 2, apply_dropout=True),  # (batch_size, 8, 8, 1024)
#     upsample(512, 2),  # (batch_size, 16, 16, 1024)
#     upsample(256, 2),  # (batch_size, 32, 32, 512)
#     upsample(128, 2),  # (batch_size, 64, 64, 256)
#     upsample(64, 2),  # (batch_size, 128, 128, 128)
#   ]

#   initializer = tf.random_normal_initializer(0., 0.02)
#   last = tf.keras.layers.Conv2DTranspose(OUTPUT_CHANNELS, 2,
#                                          strides=2,
#                                          padding='same',
#                                          kernel_initializer=initializer,
#                                          activation='tanh')  # (batch_size, 256, 256, 3)

#   x = inputs

#   # Downsampling through the model
#   skips = []
#   for down in down_stack:
#     x = down(x)
#     skips.append(x)

#   skips = reversed(skips[:-1])

#   # Upsampling and establishing the skip connections
#   for up, skip in zip(up_stack, skips):
#     x = up(x)
#     x = tf.keras.layers.Concatenate()([x, skip])

#   x = last(x)

#   return tf.keras.Model(inputs=inputs, outputs=x)

In [ ]:
def Generator():
    inputs = tf.keras.layers.Input(shape=[256, 256, 1])

    # Downsampling path (9 layers)
    down_stack = [
        downsample(64, 2, apply_batchnorm=False),  # 256x256 -> 128x128 (Layer 1)
        downsample(64, 1),                         # 128x128 -> 128x128 (Layer 2)
        downsample(128, 2),                         # 128x128 -> 64x64 (Layer 3)
        downsample(128, 1),                        # 64x64 -> 64x64 (Layer 4)
        downsample(256, 2),                         # 64x64 -> 32x32 (Layer 5)
        downsample(256, 1),                         # 32x32 -> 32x32 (Layer 6)
        downsample(512, 2),                         # 32x32 -> 16x16 (Layer 7)
        downsample(512, 1),                         # 16x16 -> 16x16 (Layer 8)
        downsample(512, 2),                         # 16x16 -> 8x8 (Layer 9)
    ]

    # Bottleneck (1 layer)
    bottleneck = downsample(512, 1)                 # 8x8 -> 8x8 (Layer 10)

    # Upsampling path (8 layers)
    up_stack = [
        upsample(512, 2, apply_dropout=True),      # 8x8 -> 16x16 (Layer 11)
        upsample(512, 1, apply_dropout=True),      # 16x16 -> 16x16 (Layer 12)
        upsample(512, 2, apply_dropout=True),      # 16x16 -> 32x32 (Layer 13)
        upsample(256, 1),                          # 32x32 -> 32x32 (Layer 14)
        upsample(256, 2),                          # 32x32 -> 64x64 (Layer 15)
        upsample(128, 1),                          # 64x64 -> 64x64 (Layer 16)
        upsample(128, 2),                          # 64x64 -> 128x128 (Layer 17)
        upsample(64, 1),                           # 128x128 -> 128x128 (Layer 18)
    ]

    # Final layer (Layer 19)
    last = tf.keras.layers.Conv2DTranspose(
        1, 2, strides=2,
        padding='same',
        kernel_initializer=tf.random_normal_initializer(0., 0.02),
        activation='tanh'                           # 128x128 -> 256x256 (Layer 19)
    )

    x = inputs
    skips = []

    # Downsampling
    for down in down_stack:
        x = down(x)
        skips.append(x)

    # Bottleneck
    x = bottleneck(x)

    # Prepare skip connections (remove the last down layer to match up layers)
    skips = reversed(skips[:-1])  # Exclude the 8x8 layer

    # Upsampling with skip connections
    for up, skip in zip(up_stack, skips):
        x = up(x)
        # Ensure spatial dimensions match before concatenation
        if x.shape[1] != skip.shape[1]:
            x = tf.keras.layers.Resizing(skip.shape[1], skip.shape[1])(x)
        x = tf.keras.layers.Concatenate()([x, skip])

    # Final upsampling
    x = last(x)

    return tf.keras.Model(inputs=inputs, outputs=x)

In [ ]:
generator = Generator()
tf.keras.utils.plot_model(generator, show_shapes=True, dpi=64)

In [ ]:
gen_output = generator(inp[tf.newaxis, ...], training=False)
plt.imshow(gen_output[0, ...], cmap='grey')
plt.colorbar()

In [ ]:
LAMBDA = 100

In [ ]:
loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)

In [ ]:
# def generator_loss(disc_generated_output, gen_output, target):
#   gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)

#   # Mean absolute error
#   l1_loss = tf.reduce_mean(tf.abs(target - gen_output))

#   total_gen_loss = gan_loss + (LAMBDA * l1_loss)

#   return total_gen_loss, gan_loss, l1_loss

def sobel_operator(image_tensor):
    """Applies the Sobel operator to an image tensor."""
    sobel_x = tf.constant([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], dtype=tf.float32)
    sobel_y = tf.constant([[1., 2., 1.], [0., 0., 0.], [-1., -2., -1.]], dtype=tf.float32)

    # Reshape kernels for conv2d
    sobel_x = tf.reshape(sobel_x, [3, 3, 1, 1])
    sobel_y = tf.reshape(sobel_y, [3, 3, 1, 1])

    # Apply Sobel kernels
    grad_x = tf.nn.conv2d(image_tensor, sobel_x, strides=[1, 1, 1, 1], padding='SAME')
    grad_y = tf.nn.conv2d(image_tensor, sobel_y, strides=[1, 1, 1, 1], padding='SAME')

    # Calculate gradient magnitude
    gradient_magnitude = tf.sqrt(tf.square(grad_x) + tf.square(grad_y))

    return gradient_magnitude

def generator_loss(disc_generated_output, gen_output, target):
  gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)

  # Mean absolute error
  l1_loss = tf.reduce_mean(tf.abs(target - gen_output))

  # Apply Sobel operator
  gen_sobel = sobel_operator(gen_output)
  target_sobel = sobel_operator(target)

  # Calculate L1 loss on Sobel filtered images
  sobel_l1_loss = tf.reduce_mean(tf.abs(target_sobel - gen_sobel))

  total_gen_loss = gan_loss + (30 * l1_loss) + (10 * sobel_l1_loss)

  return total_gen_loss, gan_loss, l1_loss

In [ ]:
# def Discriminator():
#   initializer = tf.random_normal_initializer(0., 0.02)

#   inp = tf.keras.layers.Input(shape=[256, 256, 1], name='input_image')
#   tar = tf.keras.layers.Input(shape=[256, 256, 1], name='target_image')

#   x = tf.keras.layers.concatenate([inp, tar])  # (batch_size, 256, 256, channels*2)

#   down1 = downsample(64, 4, False)(x)  # (batch_size, 128, 128, 64)
#   down2 = downsample(128, 4)(down1)  # (batch_size, 64, 64, 128)
#   down3 = downsample(256, 4)(down2)  # (batch_size, 32, 32, 256)

#   zero_pad1 = tf.keras.layers.ZeroPadding2D()(down3)  # (batch_size, 34, 34, 256)
#   conv = tf.keras.layers.Conv2D(512, 4, strides=1,
#                                 kernel_initializer=initializer,
#                                 use_bias=False)(zero_pad1)  # (batch_size, 31, 31, 512)

#   batchnorm1 = tf.keras.layers.BatchNormalization()(conv)

#   leaky_relu = tf.keras.layers.LeakyReLU()(batchnorm1)

#   zero_pad2 = tf.keras.layers.ZeroPadding2D()(leaky_relu)  # (batch_size, 33, 33, 512)

#   last = tf.keras.layers.Conv2D(1, 2, strides=1,
#                                 kernel_initializer=initializer)(zero_pad2)  # (batch_size, 30, 30, 1)

#   return tf.keras.Model(inputs=[inp, tar], outputs=last)

In [ ]:
def Discriminator():
    initializer = tf.random_normal_initializer(0., 0.02)

    inp = tf.keras.layers.Input(shape=[256, 256, 1], name='input_image')
    tar = tf.keras.layers.Input(shape=[256, 256, 1], name='target_image')

    # Layer 1: Concatenate and first conv (256x256 -> 128x128)
    x = tf.keras.layers.concatenate([inp, tar])  # (256,256,2)
    x = downsample(64, 4, apply_batchnorm=False)(x)  # Layer 1 (128,128,64)

    # Layer 2: (128x128 -> 64x64)
    x = downsample(128, 4)(x)  # Layer 2 (64,64,128)

    # Layer 3: (64x64 -> 32x32)
    x = downsample(256, 4)(x)  # Layer 3 (32,32,256)

    # Layer 4: (32x32 -> 31x31) - No stride reduction
    x = tf.keras.layers.ZeroPadding2D()(x)  # (34,34,256)
    x = tf.keras.layers.Conv2D(512, 4, strides=1,
                              kernel_initializer=initializer,
                              use_bias=False)(x)  # Layer 4 (31,31,512)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU()(x)

    # Layer 5: Final output (31x31 -> 30x30)
    x = tf.keras.layers.ZeroPadding2D()(x)  # (33,33,512)
    x = tf.keras.layers.Conv2D(1, 4, strides=1,  # Changed from 2 to 4 kernel size
                              kernel_initializer=initializer)(x)  # Layer 5 (30,30,1)

    return tf.keras.Model(inputs=[inp, tar], outputs=x)

In [ ]:
discriminator = Discriminator()
tf.keras.utils.plot_model(discriminator, show_shapes=True, dpi=64)

In [ ]:
disc_out = discriminator([inp[tf.newaxis, ...], gen_output], training=False)
plt.imshow(disc_out[0, ..., -1], cmap='grey')
plt.colorbar()

In [ ]:
def discriminator_loss(disc_real_output, disc_generated_output):
  real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)

  generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)

  total_disc_loss = real_loss + generated_loss

  return total_disc_loss

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

In [ ]:
checkpoint_dir = '../training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In [ ]:
def generate_images(model, test_input, tar):
  prediction = model(test_input, training=True)
  plt.figure(figsize=(15, 15))

  display_list = [test_input[0], tar[0], prediction[0]]
  title = ['Input Image', 'Ground Truth', 'Predicted Image']

  for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.title(title[i])
    # Getting the pixel values in the [0, 1] range to plot.
    plt.imshow(display_list[i], vmin=-.1, vmax=.1, cmap='grey')
    plt.axis('off')
  plt.show()

## Training

In [ ]:
log_dir="logs/"

summary_writer = tf.summary.create_file_writer(
  log_dir + "fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

In [ ]:
@tf.function
def train_step(input_image, target, step):
  with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
    gen_output = generator(input_image, training=True)

    disc_real_output = discriminator([input_image, target], training=True)
    disc_generated_output = discriminator([input_image, gen_output], training=True)

    gen_total_loss, gen_gan_loss, gen_l1_loss = generator_loss(disc_generated_output, gen_output, target)
    disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

  generator_gradients = gen_tape.gradient(gen_total_loss,
                                          generator.trainable_variables)
  discriminator_gradients = disc_tape.gradient(disc_loss,
                                               discriminator.trainable_variables)

  generator_optimizer.apply_gradients(zip(generator_gradients,
                                          generator.trainable_variables))
  discriminator_optimizer.apply_gradients(zip(discriminator_gradients,
                                              discriminator.trainable_variables))

  with summary_writer.as_default():
    tf.summary.scalar('gen_total_loss', gen_total_loss, step=step//100)
    tf.summary.scalar('gen_gan_loss', gen_gan_loss, step=step//100)
    tf.summary.scalar('gen_l1_loss', gen_l1_loss, step=step//100)
    tf.summary.scalar('disc_loss', disc_loss, step=step//100)

In [ ]:
def fit(train_ds, test_ds, steps):
  example_input, example_target = next(iter(test_ds.take(1)))
  start = time.time()

  for step, (input_image, target) in train_ds.repeat().take(steps).enumerate():
    if (step) % 100 == 0:
      display.clear_output(wait=True)

      if step != 0:
        print(f'Time taken for 100 steps: {time.time()-start:.2f} sec\n')

      start = time.time()

      generate_images(generator, example_input, example_target)
      print(f"Step: {step//100}")

    train_step(input_image, target, step)

    # Training step
    if (step+1) % 10 == 0:
      print('.', end='', flush=True)


    # Save (checkpoint) the model every 5k steps
    if (step + 1) % 5000 == 0:
      checkpoint.save(file_prefix=checkpoint_prefix)

In [ ]:
example_input, example_target = next(iter(test_dataset.take(1)))

In [ ]:
fit(train_dataset, test_dataset, steps=400)

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In [ ]:
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

In [ ]:
dir = '../train'
files = ['2_0005.vti', '2_0010.vti', '2_0015.vti']

for file in files:
    print(file)
    inp, re = load(f'{dir}/input_{file}', f'{dir}/real_{file}')
    generate_images(generator, inp[tf.newaxis, ...], re[tf.newaxis, ...])

In [ ]:
dir = '../sample/parallel_planes'
files = ['output_data_0_5.vti', 'output_data_0_15.vti', 'output_data_0_20.vti']

for file in files:
    print(file)
    inp, re = load(f'{dir}/coarse_grid/{file}', f'{dir}/refined_grid/{file}')
    generate_images(generator, inp[tf.newaxis, ...], re[tf.newaxis, ...])

In [ ]:
dir = './sample/semi_circle'
files = ['output_data_0_5.vti', 'output_data_0_10.vti', 'output_data_0_15.vti']

for file in files:
    print(file)
    inp, re = load(f'{dir}/coarse_grid/{file}', f'{dir}/refined_grid/{file}')
    generate_images(generator, inp[tf.newaxis, ...], re[tf.newaxis, ...])

## Generate Results

In [ ]:
def tensor_to_uniform_grid(tensor, reference_grid):
    arr = tensor.numpy()

    # Remove channel dimension if present
    if arr.ndim == 4 and arr.shape[-1] == 1:
        arr = np.squeeze(arr, axis=-1)  # shape: (Z, Y, X)

    # Transpose from (Z, Y, X) → (X, Y, Z)
    arr = arr.transpose(2, 1, 0)

    # dims = reference_grid.dimensions  # should be (X, Y, Z)
    dims = (256, 256, 1)
    if arr.shape != tuple(dims):
        raise ValueError(f"Tensor shape {arr.shape} does not match grid dimensions {dims}")

    # Build new grid
    new_grid = pv.ImageData()
    new_grid.origin = reference_grid.origin
    new_grid.spacing = reference_grid.spacing
    new_grid.dimensions = dims
    new_grid.point_data["ProcessedData"] = arr.flatten(order="F")

    return new_grid

def generate_results(model, input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        input_path = os.path.join(input_dir, filename)
        grid = pv.read(input_path)
        inp, _ = load(input_path)
        prediction = model(inp[tf.newaxis, ...], training=True)
        # prediction = tf.image.resize(prediction, [200, 200],
        #                         method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
        # print(prediction)
        output_path = os.path.join(output_dir, filename)
        processed_grid = tensor_to_uniform_grid(prediction, grid)
        processed_grid.save(output_path)
        print(f"Saved: {output_path}")

    print(f"Successfully processed all .vti files in {input_dir} into {output_dir}")


### Constant Velocity

In [ ]:
generate_results(generator, './sample/constant_vel/coarse_grid', './results/constant_vel')


### Parallel Planes

In [ ]:
generate_results(generator, './sample/parallel_planes/coarse_grid', './results/parallel_planes')